In [0]:
#Notebook 02: Reading from Staging and writing to Raw

from datetime import datetime

#Fetching values from job parameters
catalog_name    = dbutils.widgets.get("catalog_name")
source_catalog  = dbutils.widgets.get("source_catalog")
source_schema   = dbutils.widgets.get("source_schema")
meta_schema     = dbutils.widgets.get("meta_schema")
staging_schema  = dbutils.widgets.get("staging_schema")
base_path       = dbutils.widgets.get("base_path")

# Load Raw
# Reads from staging, writes Parquet to Raw Volume
# Logs results to child metrics metadata table

from datetime import datetime

#READING ACTIVE TABLES FROM PARENT METADATA

active_tables_df = spark.sql(f"""
    SELECT table_name
    FROM {catalog_name}.{meta_schema}.parent_metadata
    WHERE active_flag = 'Y'
""")

active_tables = [row["table_name"] for row in active_tables_df.collect()]
print(f"Active tables to process: {active_tables}\n")


#PROCESSING EACH TABLE

for table_name in active_tables:
    try:
        #Step 1: Read from staging
        staging_df = spark.sql(f"""
            SELECT * FROM {catalog_name}.{staging_schema}.{table_name.lower()}
        """)
        source_row_count = staging_df.count()

        #Step 2: Build dynamic timestamped file path
        execution_time = datetime.now()
        date_path      = execution_time.strftime("%Y/%m/%d")
        file_location  = f"{base_path}/{table_name.lower()}/{date_path}/{table_name.lower()}.parquet"

        #Step 3: Write to Raw Volume as Parquet
        staging_df.write \
            .mode("overwrite") \
            .parquet(file_location)

        #Step 4: Verify target row count matches source
        target_df        = spark.read.parquet(file_location)
        target_row_count = target_df.count()
        status           = "SUCCESS" if source_row_count == target_row_count else "FAILED"

        #Step 5: Log to child metrics metadata
        spark.sql(f"""
            INSERT INTO {catalog_name}.{meta_schema}.child_metrics_metadata VALUES (
                '{table_name}',
                '{execution_time}',
                '{status}',
                {source_row_count},
                {target_row_count},
                '{file_location}',
                current_date()
            )
        """)

        print(f"{table_name} | {status} | Source: {source_row_count} | Raw: {target_row_count}")
        print(f"  Written to: {file_location}")

    except Exception as e:
        #Log failure to child metrics
        import traceback
        print(f"{table_name} FAILED: {str(e)}")
        print(traceback.format_exc())  # ADD THIS — shows full stack trace
        spark.sql(f"""
            INSERT INTO {catalog_name}.{meta_schema}.child_metrics_metadata VALUES (
                '{table_name}',
                '{datetime.now()}',
                'FAILED',
                0,
                0,
                'N/A',
                current_date()
            )
    """)

print("\nRaw load completed...")